# 🤖 Banco de Pruebas en Vivo y Captura de Evidencias - LSP (Robot NAO)
### Practica N.° 2 - Inteligencia Artificial (UPC)

Este cuaderno permite ejecutar la inferencia en tiempo real del modelo **SVM (RBF Kernel)** para el reconocimiento de **Lengua de Senas Peruana (LSP)** en Google Colab.

#### Funcionalidades Implementadas:
1. **Deteccion Bimanual en Tiempo Real:** Extraccion de 126 descriptores geometricos normalizados con **MediaPipe Hands**.
2. **Disparador Automatico de Evidencias:** Captura instantanea al detectar una sena sostenida con alta certeza ($\ge 85\%$) durante 0.75 s.
3. **Estampado Diagnostico:** Cada fotograma capturado incluye la clase detectada, porcentaje de certeza, latencia y recuadro envolvente.
4. **Mosaico y Galeria Final:** Al finalizar la sesion, se despliegan automaticamente todas las capturas realizadas en un mosaico grafico.

In [ ]:
# 1. Instalacion de dependencias requeridas en Google Colab
!pip install -q mediapipe==0.10.14 opencv-python-headless scikit-learn joblib matplotlib pandas
print("Dependencias instaladas correctamente.")

In [ ]:
# 2. Carga y verificacion del modelo de clasificacion LSP serializado (.joblib)
import os
import joblib
from google.colab import files

RUTA_MODELO = "modelo_lsp_ganador.joblib"

if not os.path.exists(RUTA_MODELO):
    if os.path.exists("models/modelo_lsp_ganador.joblib"):
        RUTA_MODELO = "models/modelo_lsp_ganador.joblib"
    else:
        print("El modelo 'modelo_lsp_ganador.joblib' no se encuentra en el entorno actual.")
        print("Por favor, selecciona y sube el archivo 'modelo_lsp_ganador.joblib' desde la carpeta 'models/' de tu equipo:")
        archivos_subidos = files.upload()
        for nombre in archivos_subidos.keys():
            if "modelo_lsp_ganador" in nombre:
                RUTA_MODELO = nombre
                break

paquete = joblib.load(RUTA_MODELO)
modelo_lsp = paquete["modelo"]
nombres_clases = paquete["clases"]

print(f"\nModelo cargado con exito: {paquete.get('nombre_modelo', 'SVM')}")
print(f"Total de clases registradas: {len(nombres_clases)}")
print(f"Catalogo de senas: {nombres_clases}")

In [ ]:
# 3. Funciones de normalizacion geometrica y extraccion de descriptores bimanuales
from datetime import datetime
import time
from typing import Any
import cv2
import mediapipe as mp
import numpy as np

TOTAL_PUNTOS_POR_MANO = 21
DIMENSIONES_POR_PUNTO = 3
TOTAL_VALORES_POR_MANO = TOTAL_PUNTOS_POR_MANO * DIMENSIONES_POR_PUNTO  # 63 valores


def normalizar_puntos_mano(puntos_mano: Any) -> list[float]:
    """Calcula coordenadas 3D relativas a la muneca y normaliza la escala."""
    punto_base = puntos_mano.landmark[0]
    base_x, base_y, base_z = punto_base.x, punto_base.y, punto_base.z

    coordenadas_relativas: list[float] = []
    for punto in puntos_mano.landmark:
        coordenadas_relativas.extend([
            punto.x - base_x,
            punto.y - base_y,
            punto.z - base_z
        ])

    distancia_maxima = max(max(map(abs, coordenadas_relativas)), 1e-6)
    return [valor / distancia_maxima for valor in coordenadas_relativas]


def extraer_vectores_bimanuales(resultados: Any) -> tuple[list[float], list[float], bool, bool]:
    """Obtiene los vectores normalizados de 63 elementos para cada mano con zero-padding."""
    vector_izq = [0.0] * TOTAL_VALORES_POR_MANO
    vector_der = [0.0] * TOTAL_VALORES_POR_MANO
    detectada_izq = False
    detectada_der = False

    if not resultados.multi_hand_landmarks or not resultados.multi_handedness:
        return vector_izq, vector_der, detectada_izq, detectada_der

    for puntos, clasificacion in zip(resultados.multi_hand_landmarks, resultados.multi_handedness):
        etiqueta = clasificacion.classification[0].label
        vector_normalizado = normalizar_puntos_mano(puntos)

        if etiqueta == "Left":
            if not detectada_izq:
                vector_izq = vector_normalizado
                detectada_izq = True
            else:
                vector_der = vector_normalizado
                detectada_der = True
        elif etiqueta == "Right":
            if not detectada_der:
                vector_der = vector_normalizado
                detectada_der = True
            else:
                vector_izq = vector_normalizado
                detectada_izq = True

    return vector_izq, vector_der, detectada_izq, detectada_der


def calcular_caja_delimitadora_manos(
    resultados: Any,
    ancho_px: int,
    alto_px: int
) -> tuple[int, int, int, int] | None:
    """Calcula coordenadas envolventes de las manos para destacar la sena."""
    if not resultados.multi_hand_landmarks:
        return None

    todos_x: list[int] = []
    todos_y: list[int] = []
    for marcas_mano in resultados.multi_hand_landmarks:
        for punto in marcas_mano.landmark:
            todos_x.append(int(punto.x * ancho_px))
            todos_y.append(int(punto.y * alto_px))

    if not todos_x or not todos_y:
        return None

    margen = 25
    x_min = max(0, min(todos_x) - margen)
    y_min = max(0, min(todos_y) - margen)
    x_max = min(ancho_px - 1, max(todos_x) + margen)
    y_max = min(alto_px - 1, max(todos_y) + margen)
    return x_min, y_min, x_max, y_max


def generar_fotograma_resultado(
    fotograma_base: np.ndarray,
    clase_idx: int,
    clase_nombre: str,
    confianza: float,
    latencia_ms: float,
    caja_manos: tuple[int, int, int, int] | None
) -> np.ndarray:
    """Genera la imagen de evidencia anotada indicando la clase detectada."""
    resultado = fotograma_base.copy()
    alto_img, ancho_img, _ = resultado.shape
    marca_tiempo = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if caja_manos is not None:
        x1, y1, x2, y2 = caja_manos
        cv2.rectangle(resultado, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            resultado, f"Sena: {clase_nombre}", (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2
        )

    alto_panel = 85
    sub_img = resultado[0:alto_panel, 0:ancho_img]
    rect_oscuro = np.zeros_like(sub_img, dtype=np.uint8)
    rect_oscuro[:] = (18, 18, 18)
    cv2.addWeighted(sub_img, 0.20, rect_oscuro, 0.80, 0, sub_img)
    resultado[0:alto_panel, 0:ancho_img] = sub_img

    cv2.line(resultado, (0, alto_panel), (ancho_img, alto_panel), (0, 220, 100), 3)

    cv2.putText(
        resultado, f"CLASE DETECTADA: {clase_nombre} (ID: {clase_idx})",
        (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 255, 120), 2
    )

    detalles = f"Confianza: {confianza:.1f}% | Latencia: {latencia_ms:.2f} ms | Fecha: {marca_tiempo}"
    cv2.putText(
        resultado, detalles, (20, 68),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (230, 230, 230), 1
    )

    return resultado

print("Modulo de vision y normalizacion cargado.")

In [ ]:
# 4. Banco de pruebas interactivo con camara web en vivo (WebRTC) y captura automatica
import base64
import os
import time
from IPython.display import HTML, Javascript, display
from google.colab.output import eval_js

DIRECTORIO_EVIDENCIAS_COLAB = "evidencias_colab"
os.makedirs(DIRECTORIO_EVIDENCIAS_COLAB, exist_ok=True)

UMBRAL_CONFIANZA_AUTO = 85.0
TIEMPO_ESTABILIDAD_SEG = 0.75
ENFRIAMIENTO_MISMA_CLASE_SEG = 3.5
MAX_FOTOS_POR_CLASE = 3

lista_evidencias_colab = []

CODIGO_JAVASCRIPT_WEBCAM = """
async function iniciarFlujoCamara() {
    const contenedor = document.createElement('div');
    contenedor.style.display = 'flex';
    contenedor.style.flexDirection = 'column';
    contenedor.style.alignItems = 'center';
    contenedor.style.gap = '10px';
    contenedor.style.padding = '12px';
    contenedor.style.background = '#1e1e1e';
    contenedor.style.borderRadius = '8px';
    contenedor.style.color = '#fff';
    contenedor.style.fontFamily = 'sans-serif';

    const titulo = document.createElement('h3');
    titulo.innerText = '📹 Inferencia LSP en Vivo - Camara Web';
    titulo.style.margin = '4px';
    contenedor.appendChild(titulo);

    const video = document.createElement('video');
    video.style.borderRadius = '6px';
    video.style.transform = 'scaleX(-1)';
    video.width = 480;
    video.height = 360;
    video.autoplay = true;
    contenedor.appendChild(video);

    const panelInfo = document.createElement('div');
    panelInfo.id = 'panel_telemetria_lsp';
    panelInfo.style.fontSize = '14px';
    panelInfo.style.fontWeight = 'bold';
    panelInfo.style.color = '#00ff78';
    panelInfo.innerText = 'Iniciando detector...';
    contenedor.appendChild(panelInfo);

    const filaBotones = document.createElement('div');
    filaBotones.style.display = 'flex';
    filaBotones.style.gap = '15px';

    const btnCapturaManual = document.createElement('button');
    btnCapturaManual.innerText = '📸 Captura Manual';
    btnCapturaManual.style.padding = '8px 16px';
    btnCapturaManual.style.background = '#0284c7';
    btnCapturaManual.style.color = 'white';
    btnCapturaManual.style.border = 'none';
    btnCapturaManual.style.borderRadius = '4px';
    btnCapturaManual.style.cursor = 'pointer';

    const btnDetener = document.createElement('button');
    btnDetener.innerText = '⏹️ Detener Banco de Pruebas';
    btnDetener.style.padding = '8px 16px';
    btnDetener.style.background = '#dc2626';
    btnDetener.style.color = 'white';
    btnDetener.style.border = 'none';
    btnDetener.style.borderRadius = '4px';
    btnDetener.style.cursor = 'pointer';

    filaBotones.appendChild(btnCapturaManual);
    filaBotones.appendChild(btnDetener);
    contenedor.appendChild(filaBotones);

    document.querySelector('#output-body').appendChild(contenedor);

    const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
    video.srcObject = stream;
    await video.play();

    const canvas = document.createElement('canvas');
    canvas.width = 640;
    canvas.height = 480;
    const ctx = canvas.getContext('2d');

    window.solicitar_captura_manual = false;
    window.detener_sesion_lsp = false;

    btnCapturaManual.onclick = () => { window.solicitar_captura_manual = true; };
    btnDetener.onclick = () => {
        window.detener_sesion_lsp = true;
        stream.getTracks().forEach(track => track.stop());
        btnDetener.disabled = true;
        btnDetener.innerText = 'Sesion Detenida';
    };

    window.obtenerFotogramaWebcam = async function() {
        if (window.detener_sesion_lsp) {
            return {terminar: true, manual: false, imagen: ''};
        }
        ctx.save();
        ctx.scale(-1, 1);
        ctx.drawImage(video, -canvas.width, 0, canvas.width, canvas.height);
        ctx.restore();

        const dataUrl = canvas.toDataURL('image/jpeg', 0.85);
        const pedirManual = window.solicitar_captura_manual;
        window.solicitar_captura_manual = false;

        return {
            terminar: false,
            manual: pedirManual,
            imagen: dataUrl.split(',')[1]
        };
    };

    window.actualizarEstadoHUD = function(texto, colorHex) {
        panelInfo.innerText = texto;
        panelInfo.style.color = colorHex;
    };

    return true;
}
"""

def ejecutar_banco_pruebas_colab() -> None:
    """Ejecuta la captura de webcam, inferencia bimanual y disparo automatico en Colab."""
    global lista_evidencias_colab
    lista_evidencias_colab = []

    display(Javascript(CODIGO_JAVASCRIPT_WEBCAM))
    eval_js('iniciarFlujoCamara()')

    detector_manos = mp.solutions.hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.70,
        min_tracking_confidence=0.50
    )

    conteo_fotos: dict[int, int] = {i: 0 for i in range(len(nombres_clases))}
    ultimo_disparo: dict[int, float] = {i: 0.0 for i in range(len(nombres_clases))}

    candidata_actual: int | None = None
    tiempo_inicio_postura: float | None = None

    print("\nIniciando banco de pruebas en tiempo real...")
    print("Sostenga una sena frente a la camara para captura automatica o pulse 'Captura Manual'.")
    print("Al finalizar, pulse 'Detener Banco de Pruebas' para ver la galeria completa de resultados.\n")

    try:
        while True:
            datos_frame = eval_js('obtenerFotogramaWebcam()')
            if datos_frame.get('terminar', False):
                break

            cadena_base64 = datos_frame.get('imagen', '')
            if not cadena_base64:
                time.sleep(0.05)
                continue

            bytes_imagen = base64.b64decode(cadena_base64)
            arreglo_np = np.frombuffer(bytes_imagen, dtype=np.uint8)
            fotograma = cv2.imdecode(arreglo_np, cv2.IMREAD_COLOR)

            if fotograma is None:
                continue

            alto_px, ancho_px, _ = fotograma.shape
            fotograma_rgb = cv2.cvtColor(fotograma, cv2.COLOR_BGR2RGB)
            resultados = detector_manos.process(fotograma_rgb)

            vec_izq, vec_der, det_izq, det_der = extraer_vectores_bimanuales(resultados)
            caja_manos = calcular_caja_delimitadora_manos(resultados, ancho_px, alto_px)

            clase_idx = 0
            nombre_predicho = "EN REPOSO"
            confianza_pct = 0.0
            latencia_ms = 0.0
            tiempo_actual = time.time()
            disparar_manual = datos_frame.get('manual', False)

            if det_izq or det_der:
                vector_in = np.array(vec_izq + vec_der, dtype=np.float32).reshape(1, -1)
                t_ini = time.perf_counter()
                probabilidades = modelo_lsp.predict_proba(vector_in)[0]
                t_fin = time.perf_counter()
                latencia_ms = (t_fin - t_ini) * 1000.0

                clase_idx = int(np.argmax(probabilidades))
                confianza_pct = float(probabilidades[clase_idx] * 100.0)
                nombre_predicho = nombres_clases[clase_idx]

                # Logica de disparo automatico por estabilidad
                if confianza_pct >= UMBRAL_CONFIANZA_AUTO:
                    if candidata_actual == clase_idx and tiempo_inicio_postura is not None:
                        tiempo_sostenido = tiempo_actual - tiempo_inicio_postura
                        pct_barra = int(min(1.0, tiempo_sostenido / TIEMPO_ESTABILIDAD_SEG) * 100)

                        if tiempo_sostenido >= TIEMPO_ESTABILIDAD_SEG:
                            limite_alcanzado = conteo_fotos[clase_idx] >= MAX_FOTOS_POR_CLASE
                            enfriado = (tiempo_actual - ultimo_disparo[clase_idx]) >= ENFRIAMIENTO_MISMA_CLASE_SEG

                            if not limite_alcanzado and enfriado:
                                img_res = generar_fotograma_resultado(
                                    fotograma, clase_idx, nombre_predicho,
                                    confianza_pct, latencia_ms, caja_manos
                                )
                                ruta_foto = os.path.join(
                                    DIRECTORIO_EVIDENCIAS_COLAB,
                                    f"auto_evidencia_{clase_idx}_{nombre_predicho}_{int(tiempo_actual)}.png"
                                )
                                cv2.imwrite(ruta_foto, img_res)

                                lista_evidencias_colab.append({
                                    "fotograma": img_res,
                                    "clase_nombre": nombre_predicho,
                                    "confianza": confianza_pct,
                                    "latencia_ms": latencia_ms,
                                    "tipo": "AUTO"
                                })
                                conteo_fotos[clase_idx] += 1
                                ultimo_disparo[clase_idx] = tiempo_actual
                                tiempo_inicio_postura = None
                                print(f"[CAPTURA AUTOMATICA] {nombre_predicho} ({confianza_pct:.1f}%) guardada.")
                        else:
                            texto_hud = f"Sena: {nombre_predicho} ({confianza_pct:.1f}%) | Capturando en: {pct_barra}%"
                            eval_js(f'actualizarEstadoHUD("{texto_hud}", "#00ff78")')
                    else:
                        candidata_actual = clase_idx
                        tiempo_inicio_postura = tiempo_actual
                else:
                    candidata_actual = None
                    tiempo_inicio_postura = None
                    texto_hud = f"Sena: {nombre_predicho} ({confianza_pct:.1f}%) | Certeza moderada"
                    eval_js(f'actualizarEstadoHUD("{texto_hud}", "#facc15")')
            else:
                candidata_actual = None
                tiempo_inicio_postura = None
                eval_js('actualizarEstadoHUD("Esperando deteccion de manos...", "#94a3b8")')

            if disparar_manual:
                img_man = generar_fotograma_resultado(
                    fotograma, clase_idx, nombre_predicho,
                    confianza_pct, latencia_ms, caja_manos
                )
                ruta_foto_man = os.path.join(
                    DIRECTORIO_EVIDENCIAS_COLAB,
                    f"manual_evidencia_{clase_idx}_{nombre_predicho}_{int(tiempo_actual)}.png"
                )
                cv2.imwrite(ruta_foto_man, img_man)
                lista_evidencias_colab.append({
                    "fotograma": img_man,
                    "clase_nombre": nombre_predicho,
                    "confianza": confianza_pct,
                    "latencia_ms": latencia_ms,
                    "tipo": "MANUAL"
                })
                print(f"[CAPTURA MANUAL] {nombre_predicho} ({confianza_pct:.1f}%) guardada.")

            time.sleep(0.04)

    finally:
        detector_manos.close()
        print("\nSesion de banco de pruebas finalizada.")
        print(f"Total de evidencias capturadas: {len(lista_evidencias_colab)}")

ejecutar_banco_pruebas_colab()

In [ ]:
# 5. Despliegue de la galeria completa con todas las capturas realizadas al finalizar la sesion
import matplotlib.pyplot as plt

def mostrar_galeria_capturas_colab(evidencias: list[dict[str, Any]]) -> None:
    """Despliega todas las evidencias fotograficas capturadas en un mosaico grafico."""
    if not evidencias:
        print("\nNo se registraron capturas de evidencias durante esta sesion.")
        return

    total = len(evidencias)
    columnas = min(3, total)
    filas = (total + columnas - 1) // columnas

    plt.figure(figsize=(6 * columnas, 4.8 * filas), dpi=100)

    for idx, ev in enumerate(evidencias):
        plt.subplot(filas, columnas, idx + 1)
        img_rgb = cv2.cvtColor(ev["fotograma"], cv2.COLOR_BGR2RGB)
        plt.imshow(img_rgb)
        plt.title(
            f"Captura {idx + 1}: {ev['clase_nombre']} ({ev['confianza']:.1f}%)\n"
            f"Latencia: {ev['latencia_ms']:.1f} ms | Modo: {ev['tipo']}",
            fontsize=11, fontweight="bold", color="darkgreen"
        )
        plt.axis("off")

    plt.suptitle(
        f"Galeria Final de Evidencias Capturadas ({total} fotografias)",
        fontsize=15, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    plt.show()

mostrar_galeria_capturas_colab(lista_evidencias_colab)